In [1]:
from pathlib import Path
import numpy as np
import time, threading, queue, io

from tqdm.auto import tqdm
from astropy.io import fits
from astroquery.mast import Observations

import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont

import ipywidgets as widgets
from IPython.display import display

# ---------------- USER SETTINGS ----------------
TARGET_NAME = "TRAPPIST-1"   # try: "WASP-39", "WASP-96", "HD 209458", "GJ 1214"
INSTRUMENT  = "NIRISS"      # "NIRISS" or "NIRSPEC"
RADIUS      = "0.5 deg"     # bigger radius helps when names/coords are tricky

# What to render:
MODE = "detector"   # "detector" | "difference" | "lightcurve"
# detector    = cropped, background-subtracted detector frames
# difference  = (frame - median_frame) -> changes become obvious
# lightcurve  = plot of summed flux over time (most understandable)

# Download/render limits for the first run
MAX_OBS_TO_TRY   = 50         # how many observations to try before giving up
MAX_FILES        = 1          # set >1 to concatenate multiple files into one long video
MAX_FRAMES_TOTAL = 300        # total frames across all downloaded files

# Video + preview
FPS_VIDEO   = 12
FPS_PREVIEW = 12
PREVIEW_BUFFER_FRAMES = 180   # loops last N frames while still processing

OUT_DIR = Path("jwst_live_video")
DATA_DIR = OUT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Output:", OUT_DIR.resolve())


Output: /home/hw1970218/Desktop/fits2/jwst_live_video


In [2]:
def _subgroup_col(p):
    for c in p.colnames:
        if c.lower() in ("productsubgroupdescription", "productsubgroupdesc"):
            return c
    return None

def find_obs_rows(target_name: str, instrument: str, radius: str):
    obs = Observations.query_object(target_name, radius=radius)
    obs = obs[obs["obs_collection"] == "JWST"]

    inst = obs["instrument_name"].astype(str)
    keep = np.char.find(np.char.upper(inst), instrument.upper()) >= 0
    obs = obs[keep]
    return obs

def list_tso_products_for_obs(obs_row, prefer=("RATEINTS","CALINTS")):
    products = Observations.get_product_list(obs_row)
    p = Observations.filter_products(products, productType="SCIENCE", extension="fits")
    if len(p) == 0:
        return None

    subgroup_col = _subgroup_col(p)
    fn = p["productFilename"].astype(str)

    # prefer RATEINTS/CALINTS (great for frame-by-frame)
    for key in prefer:
        if subgroup_col and np.any(p[subgroup_col] == key):
            return p[p[subgroup_col] == key]
        if np.any(np.char.find(np.char.lower(fn), key.lower()) >= 0):
            return p[np.char.find(np.char.lower(fn), key.lower()) >= 0]

    return None

def download_one_product_row(product_row, download_dir: Path) -> Path:
    manifest = Observations.download_products(product_row[0:1], download_dir=str(download_dir), cache=True)
    local_col = "Local Path" if "Local Path" in manifest.colnames else "local_path"
    return Path(manifest[local_col][0])

def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[None, :, :]
    if a.ndim == 3:
        # common: (nints, y, x)
        if a.shape[0] <= 6000:
            return a
        # sometimes: (y, x, nints)
        if a.shape[2] <= 6000:
            return np.moveaxis(a, 2, 0)
        raise ValueError(f"Unclear 3D layout: {a.shape}")
    if a.ndim == 4:
        # common: (nints, ngroups, y, x) -> average groups
        return np.nanmean(a, axis=1)
    raise ValueError(f"Unsupported ndim={a.ndim}, shape={a.shape}")

def read_frames_from_fits(fits_path: Path):
    with fits.open(fits_path) as hdul:
        if "SCI" not in hdul:
            raise RuntimeError("No SCI extension in FITS.")
        frames = to_frame_cube(hdul["SCI"].data).astype(np.float32)

        # Optional: integration times table (not always present)
        times = None
        if "INT_TIMES" in hdul:
            tab = hdul["INT_TIMES"].data
            # try common columns
            for col in ("int_mid_MJD_UTC", "int_mid_BJD_TDB", "int_mid_BJD_TDB", "int_mid_MJD_TDB"):
                if col in tab.names:
                    times = np.array(tab[col], dtype=float)
                    break

    return frames, times

def auto_crop_bbox(frames3d, pad=20):
    """Crop around bright trace: find bounding box on median image."""
    med = np.nanmedian(frames3d, axis=0)
    m = np.nan_to_num(med, nan=np.nanmedian(med))

    # threshold at high percentile to isolate trace
    thr = np.nanpercentile(m, 99.5)
    mask = m >= thr

    if not np.any(mask):
        # fallback: no crop
        h, w = m.shape
        return (0, h, 0, w)

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()

    y0 = max(0, y0 - pad); y1 = min(m.shape[0], y1 + pad)
    x0 = max(0, x0 - pad); x1 = min(m.shape[1], x1 + pad)
    return (y0, y1, x0, x1)

def background_subtract(frame2d):
    """Simple per-frame background removal."""
    b = np.nanmedian(frame2d)
    return frame2d - b

def robust_scale_params(frames3d):
    vmin, vmax = np.nanpercentile(frames3d, [2, 99.7])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(frames3d), np.nanmax(frames3d)
    return float(vmin), float(vmax)

def scale_to_uint8(img2d, vmin, vmax, stretch="asinh"):
    x = np.nan_to_num(img2d, nan=vmin)
    x = np.clip((x - vmin) / (vmax - vmin + 1e-12), 0, 1)
    if stretch == "asinh":
        x = np.arcsinh(10 * x) / np.arcsinh(10)
    elif stretch == "log":
        x = np.log1p(1000 * x) / np.log1p(1000)
    elif stretch == "sqrt":
        x = np.sqrt(x)
    return (255 * x).astype(np.uint8)

def annotate_rgb(rgb, text):
    img = Image.fromarray(rgb)
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, img.size[0], 26], fill=(0, 0, 0))
    draw.text((6, 5), text, fill=(255, 255, 255))
    return np.array(img)

def rgb_to_png_bytes(rgb):
    img = Image.fromarray(rgb)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def make_lightcurve_frames(frames3d, times=None, width=640, height=480):
    """Return a list of RGB frames showing the running lightcurve."""
    import matplotlib.pyplot as plt

    # Sum flux in each frame (after background subtract)
    flux = np.array([np.nansum(background_subtract(f)) for f in frames3d], dtype=float)
    flux = flux - np.nanmedian(flux)

    # x-axis
    if times is None or len(times) != len(flux):
        x = np.arange(len(flux))
        xlabel = "integration index"
    else:
        x = times
        xlabel = "time (MJD/BJD units)"

    # precompute limits for stable view
    ymin, ymax = np.nanpercentile(flux, [1, 99])
    pad = 0.1 * (ymax - ymin + 1e-12)
    ymin -= pad; ymax += pad

    out = []
    for i in range(len(flux)):
        fig = plt.figure(figsize=(width/100, height/100), dpi=100)
        ax = fig.add_subplot(111)
        ax.plot(x[:i+1], flux[:i+1])
        ax.set_xlabel(xlabel)
        ax.set_ylabel("relative summed flux")
        ax.set_title("JWST TSO Lightcurve (running)")
        ax.set_ylim(ymin, ymax)
        ax.grid(True, alpha=0.3)

        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        plt.close(fig)
        out.append(img)
    return out


In [3]:
# --- Live preview widget ---
img_widget = widgets.Image(format="png")
display(img_widget)

stop_flag = threading.Event()
frame_q = queue.Queue(maxsize=400)

video_path = OUT_DIR / f"{TARGET_NAME.replace(' ','_')}_{INSTRUMENT}_{MODE}.mp4"
print("Video will be written to:", video_path)

def producer():
    try:
        obs = find_obs_rows(TARGET_NAME, INSTRUMENT, RADIUS)
        if len(obs) == 0:
            raise RuntimeError(f"No JWST {INSTRUMENT} observations found for '{TARGET_NAME}' (radius {RADIUS}).")

        # Try observations until we find products with RATEINTS/CALINTS
        files_done = 0
        frames_written = 0

        with imageio.get_writer(video_path, fps=FPS_VIDEO, codec="libx264", quality=8) as writer:
            for o in obs[:MAX_OBS_TO_TRY]:
                if files_done >= MAX_FILES or frames_written >= MAX_FRAMES_TOTAL:
                    break

                prods = list_tso_products_for_obs(o, prefer=("RATEINTS","CALINTS"))
                if prods is None or len(prods) == 0:
                    continue

                # download first product from this obs
                fits_path = download_one_product_row(prods[0:1], DATA_DIR)
                print("Downloaded:", fits_path.name)

                frames, times = read_frames_from_fits(fits_path)

                # basic crop + background subtract
                y0,y1,x0,x1 = auto_crop_bbox(frames, pad=30)
                cropped = frames[:, y0:y1, x0:x1]
                cropped = np.array([background_subtract(f) for f in cropped], dtype=np.float32)

                if MODE == "difference":
                    med = np.nanmedian(cropped, axis=0)
                    cropped = cropped - med

                # choose stable scaling for this file
                vmin, vmax = robust_scale_params(cropped)

                # optional: if "lightcurve" mode, ignore detector frames and render plot frames
                if MODE == "lightcurve":
                    lc_frames = make_lightcurve_frames(cropped, times=times)
                    for i, rgb in enumerate(lc_frames):
                        if frames_written >= MAX_FRAMES_TOTAL: break
                        text = f"{TARGET_NAME} {INSTRUMENT} | frame {frames_written}"
                        rgb2 = annotate_rgb(rgb, text)
                        writer.append_data(rgb2)
                        # push to preview queue
                        try:
                            frame_q.put_nowait(rgb_to_png_bytes(rgb2))
                        except queue.Full:
                            pass
                        frames_written += 1
                    files_done += 1
                    continue

                # detector/difference frames → video
                n = min(len(cropped), MAX_FRAMES_TOTAL - frames_written)
                for i in range(n):
                    u8 = scale_to_uint8(cropped[i], vmin, vmax, stretch="asinh")
                    rgb = np.repeat(u8[..., None], 3, axis=2)

                    # annotate
                    ttxt = ""
                    if times is not None and i < len(times):
                        ttxt = f" t={times[i]:.6f}"
                    text = f"{TARGET_NAME} {INSTRUMENT} {MODE} | frame {frames_written}{ttxt}"
                    rgb = annotate_rgb(rgb, text)

                    writer.append_data(rgb)
                    try:
                        frame_q.put_nowait(rgb_to_png_bytes(rgb))
                    except queue.Full:
                        pass

                    frames_written += 1
                    if frames_written >= MAX_FRAMES_TOTAL:
                        break

                files_done += 1

        print("Rendering finished.")
    finally:
        stop_flag.set()

def consumer():
    buffer = []
    idx = 0
    while not (stop_flag.is_set() and frame_q.empty() and len(buffer) > 0):
        # drain new frames into rolling buffer
        while True:
            try:
                b = frame_q.get_nowait()
                buffer.append(b)
                if len(buffer) > PREVIEW_BUFFER_FRAMES:
                    buffer = buffer[-PREVIEW_BUFFER_FRAMES:]
            except queue.Empty:
                break

        # loop playback of what we have so far
        if buffer:
            img_widget.value = buffer[idx % len(buffer)]
            idx += 1

        time.sleep(1.0 / FPS_PREVIEW)

    # keep looping the last buffer for a short moment at end
    for _ in range(FPS_PREVIEW * 2):
        if buffer:
            img_widget.value = buffer[idx % len(buffer)]
            idx += 1
        time.sleep(1.0 / FPS_PREVIEW)

# start threads
t_prod = threading.Thread(target=producer, daemon=True)
t_cons = threading.Thread(target=consumer, daemon=True)
t_prod.start()
t_cons.start()


Image(value=b'')

Video will be written to: jwst_live_video/TRAPPIST-1_NIRISS_detector.mp4


In [4]:
# --- Live preview widget ---
img_widget = widgets.Image(format="png")
display(img_widget)

stop_flag = threading.Event()
frame_q = queue.Queue(maxsize=400)

video_path = OUT_DIR / f"{TARGET_NAME.replace(' ','_')}_{INSTRUMENT}_{MODE}.mp4"
print("Video will be written to:", video_path)

def producer():
    try:
        obs = find_obs_rows(TARGET_NAME, INSTRUMENT, RADIUS)
        if len(obs) == 0:
            raise RuntimeError(f"No JWST {INSTRUMENT} observations found for '{TARGET_NAME}' (radius {RADIUS}).")

        # Try observations until we find products with RATEINTS/CALINTS
        files_done = 0
        frames_written = 0

        with imageio.get_writer(video_path, fps=FPS_VIDEO, codec="libx264", quality=8) as writer:
            for o in obs[:MAX_OBS_TO_TRY]:
                if files_done >= MAX_FILES or frames_written >= MAX_FRAMES_TOTAL:
                    break

                prods = list_tso_products_for_obs(o, prefer=("RATEINTS","CALINTS"))
                if prods is None or len(prods) == 0:
                    continue

                # download first product from this obs
                fits_path = download_one_product_row(prods[0:1], DATA_DIR)
                print("Downloaded:", fits_path.name)

                frames, times = read_frames_from_fits(fits_path)

                # basic crop + background subtract
                y0,y1,x0,x1 = auto_crop_bbox(frames, pad=30)
                cropped = frames[:, y0:y1, x0:x1]
                cropped = np.array([background_subtract(f) for f in cropped], dtype=np.float32)

                if MODE == "difference":
                    med = np.nanmedian(cropped, axis=0)
                    cropped = cropped - med

                # choose stable scaling for this file
                vmin, vmax = robust_scale_params(cropped)

                # optional: if "lightcurve" mode, ignore detector frames and render plot frames
                if MODE == "lightcurve":
                    lc_frames = make_lightcurve_frames(cropped, times=times)
                    for i, rgb in enumerate(lc_frames):
                        if frames_written >= MAX_FRAMES_TOTAL: break
                        text = f"{TARGET_NAME} {INSTRUMENT} | frame {frames_written}"
                        rgb2 = annotate_rgb(rgb, text)
                        writer.append_data(rgb2)
                        # push to preview queue
                        try:
                            frame_q.put_nowait(rgb_to_png_bytes(rgb2))
                        except queue.Full:
                            pass
                        frames_written += 1
                    files_done += 1
                    continue

                # detector/difference frames → video
                n = min(len(cropped), MAX_FRAMES_TOTAL - frames_written)
                for i in range(n):
                    u8 = scale_to_uint8(cropped[i], vmin, vmax, stretch="asinh")
                    rgb = np.repeat(u8[..., None], 3, axis=2)

                    # annotate
                    ttxt = ""
                    if times is not None and i < len(times):
                        ttxt = f" t={times[i]:.6f}"
                    text = f"{TARGET_NAME} {INSTRUMENT} {MODE} | frame {frames_written}{ttxt}"
                    rgb = annotate_rgb(rgb, text)

                    writer.append_data(rgb)
                    try:
                        frame_q.put_nowait(rgb_to_png_bytes(rgb))
                    except queue.Full:
                        pass

                    frames_written += 1
                    if frames_written >= MAX_FRAMES_TOTAL:
                        break

                files_done += 1

        print("Rendering finished.")
    finally:
        stop_flag.set()

def consumer():
    buffer = []
    idx = 0
    while not (stop_flag.is_set() and frame_q.empty() and len(buffer) > 0):
        # drain new frames into rolling buffer
        while True:
            try:
                b = frame_q.get_nowait()
                buffer.append(b)
                if len(buffer) > PREVIEW_BUFFER_FRAMES:
                    buffer = buffer[-PREVIEW_BUFFER_FRAMES:]
            except queue.Empty:
                break

        # loop playback of what we have so far
        if buffer:
            img_widget.value = buffer[idx % len(buffer)]
            idx += 1

        time.sleep(1.0 / FPS_PREVIEW)

    # keep looping the last buffer for a short moment at end
    for _ in range(FPS_PREVIEW * 2):
        if buffer:
            img_widget.value = buffer[idx % len(buffer)]
            idx += 1
        time.sleep(1.0 / FPS_PREVIEW)

# start threads
t_prod = threading.Thread(target=producer, daemon=True)
t_cons = threading.Thread(target=consumer, daemon=True)
t_prod.start()
t_cons.start()


Image(value=b'')

Video will be written to: jwst_live_video/TRAPPIST-1_NIRISS_detector.mp4


In [5]:
# Wait for completion
t_prod.join()
t_cons.join()

print("Saved video:", video_path.resolve())


Downloaded: jw02589002001_04102_00001-seg001_nis_rateints.fits


/tmp/ipykernel_7975/2932114196.py:76: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(frames3d, axis=0)
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (444, 91) to (448, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Rendering finished.
Saved video: /home/hw1970218/Desktop/fits2/jwst_live_video/TRAPPIST-1_NIRISS_detector.mp4


In [7]:
from IPython.display import Video, display
display(Video(str(video_path), embed=True))
